# Chapter 24 — Feature Engineering, Encoding, and Pipelines

*From Absolute Zero* — companion notebook.

Every block below is the code printed in the chapter, in the same order. Run the cells top to bottom; the output should match the book exactly. If it does not, check `requirements.txt` first, then `docs/TROUBLESHOOTING.md`.

In [1]:
!pip -q install -r https://raw.githubusercontent.com/FromAbsoluteZero/CodeBase/main/requirements.txt  # Colab only; skip locally

zsh:1: command not found: pip


## Create the data

Run once. Every dataset in this book is generated by code you can read — nothing is downloaded, so nothing can rot behind a dead link. This is the printed block from Chapter 24 (`code/ch24/gen_orders.py` in the repository).

In [2]:
import numpy as np, pandas as pd
rng = np.random.default_rng(24)
n = 8000

city = rng.choice([f"CITY_{i:03d}" for i in range(180)], n)      # high cardinality
plan = rng.choice(["basic", "plus", "pro"], n, p=[.55, .32, .13])
channel = rng.choice(["web", "app", "phone"], n, p=[.5, .38, .12])
signup = pd.to_datetime("2023-01-01") + pd.to_timedelta(
    rng.integers(0, 730, n), unit="D")
income = np.round(np.exp(rng.normal(10.2, 0.55, n)), 0)
sessions = rng.poisson(6, n)
basket = np.round(np.exp(rng.normal(3.1, 0.7, n)), 2)

# churn depends on plan, engagement, and value-for-money -- not on city
z = (-0.4
     - 0.55 * (plan == "pro") + 0.35 * (plan == "basic")
     - 0.09 * sessions
     + 1.85 * (basket / (income / 1000) > 1.4)
     + 0.30 * (channel == "phone")
     + rng.normal(0, 0.6, n))
churn = (rng.random(n) < 1 / (1 + np.exp(-z))).astype(int)

df = pd.DataFrame({"City": city, "Plan": plan, "Channel": channel,
                   "SignupDate": signup.strftime("%Y-%m-%d"),
                   "AnnualIncome": income, "Sessions": sessions,
                   "AvgBasket": basket, "Churn": churn})
df.loc[rng.random(n) < 0.09, "AnnualIncome"] = np.nan     # real gaps
df.to_csv("customers.csv", index=False)
print(f"wrote customers.csv: {n:,} customers, churn {churn.mean():.1%}, "
      f"{df.City.nunique()} cities, {df.AnnualIncome.isna().sum()} missing incomes")

wrote customers.csv: 8,000 customers, churn 44.8%, 180 cities, 720 missing incomes


## Shared setup

Imports and the objects the blocks below reuse. The chapter prints these once and then continues the same session. This cell is `code/ch24/_lib.py`.

In [3]:
import numpy as np, pandas as pd, warnings; warnings.filterwarnings("ignore")
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (StandardScaler, OneHotEncoder,
                                   OrdinalEncoder, TargetEncoder)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold, train_test_split
# customers.csv is created by this chapter's Step 1, gen_orders.py. The blocks read it from the
# working directory exactly as the book does; if it is not here yet, use the copy shipped in
# data/generated/ (byte-identical to what the generator writes).
import os as _os, shutil as _shutil
if not _os.path.exists("customers.csv"):
    for _d in ("../../data/generated", "../data/generated", "data/generated"):
        if _os.path.exists(_os.path.join(_d, "customers.csv")):
            _shutil.copy(_os.path.join(_d, "customers.csv"), "customers.csv"); break
df = pd.read_csv("customers.csv", parse_dates=["SignupDate"])
y = df.pop("Churn").values
cv = StratifiedKFold(5, shuffle=True, random_state=0)
NUM = ["AnnualIncome", "Sessions", "AvgBasket"]
LOWCARD = ["Plan", "Channel"]

## The chapter code

### Block 1  (`c1.py`)

In [4]:
print(df.dtypes.to_string())
print(f"\nrows {len(df):,}   churn {y.mean():.1%}")
for c in ["City", "Plan", "Channel"]:
    print(f"  {c:<9} {df[c].nunique():>4} distinct   "
          f"most common: {df[c].value_counts().index[0]}")
print(f"  AnnualIncome missing: {df.AnnualIncome.isna().sum()} "
      f"({df.AnnualIncome.isna().mean():.1%})")

City                       str
Plan                       str
Channel                    str
SignupDate      datetime64[us]
AnnualIncome           float64
Sessions                 int64
AvgBasket              float64

rows 8,000   churn 44.8%
  City       180 distinct   most common: CITY_137
  Plan         3 distinct   most common: basic
  Channel      3 distinct   most common: web
  AnnualIncome missing: 720 (9.0%)


### Block 2  (`c2.py`)

In [5]:
# Three ways to encode a category, compared on the same folds.
def build(cat_step, cat_cols):
    num = Pipeline([("imp", SimpleImputer(strategy="median")),
                    ("sc", StandardScaler())])
    return Pipeline([
        ("pre", ColumnTransformer([("n", num, NUM),
                                   ("c", cat_step, cat_cols)],
                                  remainder="drop")),
        ("clf", LogisticRegression(max_iter=2000))])

opts = {
  "drop City entirely":  (OneHotEncoder(handle_unknown="ignore"), LOWCARD),
  "one-hot everything":  (OneHotEncoder(handle_unknown="ignore"),
                          LOWCARD + ["City"]),
  "ordinal-code City":   (OrdinalEncoder(handle_unknown="use_encoded_value",
                          unknown_value=-1), LOWCARD + ["City"]),
  "target-encode City":  (TargetEncoder(random_state=0), LOWCARD + ["City"]),
}
print(f"{'encoding':<22}{'CV AUC':>9}{'sd':>8}{'columns':>10}")
for name, (step, cols) in opts.items():
    p = build(step, cols)
    s = cross_val_score(p, df, y, cv=cv, scoring="roc_auc")
    p.fit(df, y)
    ncol = p.named_steps["pre"].transform(df).shape[1]
    print(f"{name:<22}{s.mean():>9.4f}{s.std():>8.4f}{ncol:>10}")

encoding                 CV AUC      sd   columns
drop City entirely       0.6711  0.0165         9
one-hot everything       0.6503  0.0136       189


ordinal-code City        0.6700  0.0168         6
target-encode City       0.6711  0.0172         6


### Block 3  (`c3.py`)

In [6]:
# Target encoding replaces a category with the mean outcome for that
# category. Done naively it hands the model the answer.
Xtr, Xte, ytr, yte = train_test_split(df, y, test_size=0.3,
                                      random_state=0, stratify=y)

# WRONG: compute the means on all training rows, then use them as a feature
means = pd.Series(ytr, index=Xtr.index).groupby(Xtr["City"]).mean()
tr_naive = Xtr["City"].map(means).values
te_naive = Xte["City"].map(means).fillna(ytr.mean()).values

from sklearn.metrics import roc_auc_score
m = LogisticRegression(max_iter=2000).fit(tr_naive.reshape(-1, 1), ytr)
print("naive target encoding, City alone:")
auc_tr = roc_auc_score(ytr, m.predict_proba(tr_naive.reshape(-1, 1))[:, 1])
auc_te = roc_auc_score(yte, m.predict_proba(te_naive.reshape(-1, 1))[:, 1])
print(f"  AUC on the training rows: {auc_tr:.4f}")
print(f"  AUC on held-out rows:     {auc_te:.4f}")

# RIGHT: sklearn's TargetEncoder cross-fits internally
te = TargetEncoder(random_state=0)
tr_cf = te.fit_transform(Xtr[["City"]], ytr)
te_cf = te.transform(Xte[["City"]])
m2 = LogisticRegression(max_iter=2000).fit(tr_cf, ytr)
print("cross-fitted target encoding, City alone:")
print(f"  AUC on the training rows: "
      f"{roc_auc_score(ytr, m2.predict_proba(tr_cf)[:,1]):.4f}")
print(f"  AUC on held-out rows:     "
      f"{roc_auc_score(yte, m2.predict_proba(te_cf)[:,1]):.4f}")
print(f"\nreminder: City was generated independently of churn.")

naive target encoding, City alone:
  AUC on the training rows: 0.5875
  AUC on held-out rows:     0.5265
cross-fitted target encoding, City alone:
  AUC on the training rows: 0.5270
  AUC on held-out rows:     0.4738

reminder: City was generated independently of churn.


### Block 4  (`c4.py`)

In [7]:
# Whether a constructed feature helps depends on the model that gets it.
# The churn rule contains a threshold on basket-to-income, which a linear
# model can only approximate and a tree can express exactly.
def add_features(d):
    d = d.copy()
    d["TenureDays"] = (pd.Timestamp("2025-01-01") - d["SignupDate"]).dt.days
    d["SignupMonth"] = d["SignupDate"].dt.month
    d["BasketPerIncome"] = d["AvgBasket"] / (d["AnnualIncome"] / 1000)
    d["SessionsPerMonth"] = d["Sessions"] / (d["TenureDays"] / 30 + 1)
    return d.drop(columns=["SignupDate", "City"])

NUM2 = NUM + ["TenureDays", "SignupMonth", "BasketPerIncome",
              "SessionsPerMonth"]
d2 = add_features(df)

def build(cols, clf):
    num = Pipeline([("imp", SimpleImputer(strategy="median")),
                    ("sc", StandardScaler())])
    return Pipeline([("pre", ColumnTransformer([
                        ("n", num, cols),
                        ("c",
                         OneHotEncoder(handle_unknown="ignore"), LOWCARD)])),
                     ("clf", clf)])

print(f"{'model':<22}{'raw':>9}{'engineered':>13}{'gain':>9}")
for name, clf in [("logistic regression", LogisticRegression(max_iter=2000)),
                  ("gradient boosting",
                   HistGradientBoostingClassifier(random_state=0))]:
    a = cross_val_score(build(NUM, clf), d2, y, cv=cv,
                        scoring="roc_auc").mean()
    b = cross_val_score(build(NUM2, clf), d2, y, cv=cv,
                        scoring="roc_auc").mean()
    print(f"{name:<22}{a:>9.4f}{b:>13.4f}{b-a:>+9.4f}")

model                       raw   engineered     gain


logistic regression      0.6711       0.6770  +0.0060


gradient boosting        0.6755       0.6760  +0.0005


### Block 5  (`c5.py`)

In [8]:
# What happens in production when a category appears that training never saw.
Xtr, Xte, ytr, yte = train_test_split(df, y, test_size=0.3,
                                      random_state=0, stratify=y)
Xte = Xte.copy()
Xte.loc[Xte.index[:40], "Channel"] = "kiosk"      # a new channel launches

num = Pipeline([("imp", SimpleImputer(strategy="median")),
                ("sc", StandardScaler())])
for policy in ("error", "ignore"):
    enc = OneHotEncoder(handle_unknown=policy)
    p = Pipeline([("pre", ColumnTransformer([("n", num, NUM),
                                             ("c", enc, LOWCARD)])),
                  ("clf", LogisticRegression(max_iter=2000))]).fit(Xtr, ytr)
    try:
        pred = p.predict_proba(Xte)[:, 1]
        print(f"handle_unknown='{policy}': scored "
              f"{len(pred):,} rows without error")
    except Exception as e:
        msg = str(e).splitlines()[0]
        print(f"handle_unknown='{policy}': {type(e).__name__}: "
              f"{msg[:40].rsplit(' ', 1)[0]} ...")

handle_unknown='error': ValueError: Found unknown categories ['kiosk'] in ...
handle_unknown='ignore': scored 2,400 rows without error
